# Notebook 22: LangGraph - Error Handling & Resilience

**🎯 Goal:** To learn how to build robust, production-ready agents that can handle failures gracefully using retries, fallbacks, and timeouts.

## 🧩 Concept Overview

In the real world, things fail. APIs go down, LLMs return errors, and tools time out. A resilient agent must be able to handle these situations without crashing.

- **Retry Logic:** If a step fails, try it again a few times before giving up.
- **Graceful Degradation:** If a primary method fails, have a backup plan (e.g., use a different tool or a simpler LLM).
- **Timeouts:** Don't let a single step hang the entire workflow indefinitely.

### 7.1. Retry Logic

We can implement retry logic by adding an `error_count` to our state. If a node fails, we increment the counter and route back to the same node. If the counter exceeds a threshold, we give up and route to an error-handling node.

In [ ]:
from typing import TypedDict, Annotated
import operator
import random
from langgraph.graph import StateGraph, END

class RetryState(TypedDict):
    task: str
    result: str
    retry_count: int

def unreliable_tool_node(state: RetryState) -> dict:
    print(f"--- Attempting to use unreliable tool (Attempt {state['retry_count']}) ---")
    if random.random() < 0.5: # 50% chance of failure
        print("Tool failed!")
        return {"result": "failure", "retry_count": state['retry_count'] + 1}
    else:
        print("Tool succeeded!")
        return {"result": "success"}

def handle_failure_node(state: RetryState) -> dict:
    print("--- Tool failed after multiple retries. Giving up. ---")
    return {"result": "gave up"}

def retry_router(state: RetryState) -> str:
    if state["result"] == "success":
        return "end"
    if state["retry_count"] >= 3:
        return "handle_failure"
    return "retry"

workflow = StateGraph(RetryState)
workflow.add_node("tool", unreliable_tool_node)
workflow.add_node("handle_failure", handle_failure_node)

workflow.set_entry_point("tool")
workflow.add_edge("handle_failure", END)

workflow.add_conditional_edges(
    "tool",
    retry_router,
    {
        "retry": "tool", # Loop back to the tool node
        "handle_failure": "handle_failure",
        "end": END
    }
)

app = workflow.compile()
app.invoke({"task": "do something", "retry_count": 0})

### 7.2. Graceful Degradation

If a high-quality, expensive model fails, you might want to fall back to a cheaper, more reliable one. You can model this with a conditional edge that checks for a specific type of failure.

In [ ]:
class FallbackState(TypedDict):
    question: str
    answer: str
    model_used: str

def premium_model_node(state: FallbackState) -> dict:
    print("--- Trying premium model (GPT-4o) ---")
    # Simulate an API error
    return {"answer": "API_ERROR"}

def standard_model_node(state: FallbackState) -> dict:
    print("--- Falling back to standard model (GPT-3.5) ---")
    return {"answer": "This is a fallback answer.", "model_used": "gpt-3.5"}

def fallback_router(state: FallbackState) -> str:
    if state["answer"] == "API_ERROR":
        return "fallback"
    return "end"

fallback_workflow = StateGraph(FallbackState)
fallback_workflow.add_node("premium_model", premium_model_node)
fallback_workflow.add_node("standard_model", standard_model_node)
fallback_workflow.set_entry_point("premium_model")
fallback_workflow.add_conditional_edges("premium_model", fallback_router, {"fallback": "standard_model", "end": END})
fallback_workflow.add_edge("standard_model", END)

fallback_app = fallback_workflow.compile()
result = fallback_app.invoke({"question": "..."})
print(f"\n--- Final Result ---\n{result}")

### 7.3. Timeout & Circuit Breakers

To prevent a node from hanging, you can wrap its execution in a timeout. `asyncio.wait_for()` is a great way to do this for async nodes.

In [ ]:
import asyncio

async def slow_tool():
    print("--- Slow tool started ---")
    await asyncio.sleep(5) # This will take 5 seconds
    return "done"

async def timeout_node(state: dict) -> dict:
    try:
        result = await asyncio.wait_for(slow_tool(), timeout=2.0)
        return {"result": result}
    except asyncio.TimeoutError:
        print("--- Tool timed out! ---")
        return {"result": "timeout_error"}

# In a Jupyter environment, you can run this with:
# await timeout_node({})
print("Timeout node is defined. Uncomment the line above to test.")

## 🏁 Summary

Building resilient agents is key to making them useful in the real world.

1.  **Plan for Failure:** Design your state and graph with error handling in mind from the start.
2.  **Use State to Track Errors:** Add fields like `retry_count` or `last_error` to your state to control your error-handling logic.
3.  **Combine Patterns:** A robust agent might use all three patterns: it could *retry* a call a few times, then *fall back* to a different tool, all while wrapped in a *timeout*.

Next, we'll discuss how to integrate our LangGraph agents with a frontend and how to log and trace their execution.